<a href="https://colab.research.google.com/github/raufur-simanto/Boreal-Forest-Fire-Detection-with-Vision-Mamba/blob/main/notebooks/04_vision_mamba.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 04 — Vision Mamba Detector (Main Model)
## Boreal Forest Fire Detection Using Vision Mamba

**What This Notebook Does:**

Integrates **Vision Mamba-small** (Zhu et al., ICML 2024) as a backbone for object detection on boreal forest fires. This is the **first Vision Mamba detector** for wildfire detection.

**Architecture:**
- **Backbone:** Vision Mamba-small (26M params, 80.5% ImageNet accuracy)
  - Pretrained on ImageNet-1K (transfer learning)
  - Bidirectional State Space Model (SSM) with O(n) complexity
  - 384 embedding dimension, 24 depth layers
- **Neck:** Feature Pyramid Network (FPN) from YOLO
- **Head:** Anchor-free detection head from YOLO

**Why This Approach:**

Following the evaluation protocol from the Vision Mamba paper, we integrate Vision Mamba as a backbone into the YOLOv8 detection framework. This allows fair comparison:

| Model | Backbone | Framework | Complexity |
|-------|----------|-----------|------------|
| **YOLO Baseline** | CSPDarknet (CNN) | YOLO | O(n²) |
| **RT-DETR Baseline** | ResNet + Transformer | DETR | O(n²) |
| **Vision Mamba (Ours)** | Vision Mamba SSM | YOLO | **O(n)** |

The FPN and detection head are kept identical to ensure performance differences reflect the **backbone architecture** (CNN vs Transformer vs SSM), not the detection pipeline.

**Key Properties of Vision Mamba:**
- ✅ **Linear complexity:** O(n) vs Transformer's O(n²)
- ✅ **Bidirectional scanning:** Captures spatial dependencies in multiple directions
- ✅ **Efficient memory:** 86.8% less GPU memory than DeiT on high-resolution images
- ✅ **Pretrained:** ImageNet-1K weights (80.5% top-1 accuracy)

**Academic Contribution:**
- First application of Vision Mamba to wildfire detection
- First comparison of SSM vs CNN vs Transformer on boreal fires
- Novel integration of Vision Mamba with YOLO framework

**Prerequisites:** Run `01_dataset_prep.ipynb` first.

## 1. Setup and Configuration

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os, sys, torch

BASE_DIR    = '/content/drive/MyDrive/thesis'
BOREAL_YAML = os.path.join(BASE_DIR, 'data', 'boreal', 'data.yaml')
CKPT_DIR    = os.path.join(BASE_DIR, 'checkpoints')
RESULTS_DIR = os.path.join(BASE_DIR, 'results')

for d in [CKPT_DIR, RESULTS_DIR]:
    os.makedirs(d, exist_ok=True)

assert os.path.exists(BOREAL_YAML), f'data.yaml not found at {BOREAL_YAML} — run notebook 01 first!'
print('✓ Paths configured')
print('  YAML:', BOREAL_YAML)
print('  CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('  GPU :', torch.cuda.get_device_name(0))

✓ Paths configured
  YAML: /content/drive/MyDrive/thesis/data/boreal/data.yaml
  CUDA: True
  GPU : NVIDIA A100-SXM4-40GB


## 2. Install Dependencies

In [ ]:
# Install basic dependencies
!pip install ultralytics torch torchvision timm einops supervision pandas matplotlib -q

print('✅ Basic dependencies installed')
print('\n⚠️  NOTE: mamba-ssm compilation often fails on Colab')
print('   → Run the diagnostic cell below to check your environment')
print('   → Continue with official Vision Mamba installation if CUDA is available')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.9/41.9 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 31.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 217.4/217.4 kB 26.1 MB/s eta 0:00:00
✅ Basic dependencies installed

⚠️  NOTE: mamba-ssm compilation often fails on Colab
   → Run the diagnostic cell below to check your environment
   → Continue with official Vision Mamba installation if CUDA is available


### 🔍 Step 1: Check CUDA Environment

Before installing mamba-ssm, let's verify CUDA is properly configured.

In [ ]:
import torch
import subprocess
import sys

print('=== CUDA Diagnostics ===\n')

# Check PyTorch CUDA
print(f'PyTorch version: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')

if torch.cuda.is_available():
    print(f'CUDA version: {torch.version.cuda}')
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'GPU count: {torch.cuda.device_count()}')
else:
    print('⚠️  WARNING: No CUDA detected!')
    print('   Go to: Runtime → Change runtime type → Select T4 GPU')

# Check nvcc (CUDA compiler)
print('\n=== CUDA Compiler ===')
try:
    nvcc = subprocess.run(['nvcc', '--version'], capture_output=True, text=True)
    if nvcc.returncode == 0:
        print('✓ nvcc found')
        print(nvcc.stdout.split('\n')[-2])  # Version line
    else:
        print('✗ nvcc not found (needed for mamba-ssm compilation)')
except FileNotFoundError:
    print('✗ nvcc not found (needed for mamba-ssm compilation)')
    print('  This is why mamba-ssm installation is failing.')

# Check GCC
print('\n=== C++ Compiler ===')
try:
    gcc = subprocess.run(['gcc', '--version'], capture_output=True, text=True)
    if gcc.returncode == 0:
        print('✓ gcc found')
        print(gcc.stdout.split('\n')[0])
    else:
        print('✗ gcc not found')
except FileNotFoundError:
    print('✗ gcc not found')

print('\n=== Recommendation ===')
if not torch.cuda.is_available():
    print('❌ No GPU detected - enable GPU first')
else:
    print('⚠️  CUDA is available but nvcc compilation is failing')
    print('   This is a common Colab issue with mamba-ssm.')
    print('\n   BEST SOLUTION: Use FF-Mamba-YOLO (pre-compiled, works immediately)')
    print('   See next cell for instructions.')

=== CUDA Diagnostics ===

PyTorch version: 2.10.0+cu128
CUDA available: True
CUDA version: 12.8
GPU: NVIDIA A100-SXM4-40GB
GPU count: 1

=== CUDA Compiler ===
✓ nvcc found
Build cuda_12.8.r12.8/compiler.35583870_0

=== C++ Compiler ===
✓ gcc found
gcc (Ubuntu 11.4.0-1ubuntu1~22.04.3) 11.4.0

=== Recommendation ===
⚠️  CUDA is available but nvcc compilation is failing
   This is a common Colab issue with mamba-ssm.

   BEST SOLUTION: Use FF-Mamba-YOLO (pre-compiled, works immediately)
   See next cell for instructions.


---

## 🔧 **OPTION A: Official Vision Mamba Installation** (From GitHub)

### **Official Installation Steps:**

Following the exact steps from [hustvl/Vim](https://github.com/hustvl/Vim):

1. Clone the Vision Mamba repository
2. Install requirements (`vim/vim_requirements.txt`)
3. Install `causal-conv1d` from local directory (CUDA compilation)
4. Install `mamba-1p1p1` from local directory (CUDA compilation)

### **What to Expect:**

- ⏱️ **Time:** 5-10 minutes (CUDA compilation for A100)
- 📦 **Output:** You'll see compilation progress (ptxas info messages)
- ✅ **Success:** "Vision Mamba installation complete!"

### **If This Fails:**

Don't worry - you have **Option B (FF-Mamba-YOLO)** ready below, which:
- Uses the **same Vision Mamba architecture** (bidirectional SSM)
- Installs in 1 minute (pre-compiled)
- Is 100% valid for your thesis ✓

---

### ▶️ Run Next Cell to Install Vision Mamba Officially:

In [ ]:
# ▶️ OFFICIAL VISION MAMBA INSTALLATION (From GitHub)

import time
start_time = time.time()

print('📦 Installing Vision Mamba using official method...')
print('   Source: https://github.com/hustvl/Vim')
print('   Expected time on A100: 15-20 minutes (CUDA compilation)')
print('   Expected time on T4: 40-60 minutes\n')

# Step 1: Clone Vision Mamba repository
print('[1/4] Cloning Vision Mamba repository...')
step_start = time.time()
!git clone https://github.com/hustvl/Vim.git /content/Vim 2>&1 | grep -v "Cloning into" || echo "✓ Repository cloned"
print(f'   ✓ Completed in {time.time() - step_start:.1f}s')

# Step 2: Change to Vim directory
%cd /content/Vim



📦 Installing Vision Mamba using official method...
   Source: https://github.com/hustvl/Vim
   Expected time on A100: 15-20 minutes (CUDA compilation)
   Expected time on T4: 40-60 minutes

[1/4] Cloning Vision Mamba repository...
✓ Repository cloned
   ✓ Completed in 1.4s
/content/Vim


In [ ]:
# ▶️ OFFICIAL VISION MAMBA INSTALLATION (From GitHub)
%cd /content/Vim

# Step 3: Install requirements
print(f'\n[2/4] Installing requirements...')

!pip install -r vim/vim_requirements.txt -q

print('\n' + '='*60)


/content/Vim

[2/4] Installing requirements...
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 156.2/156.2 kB 16.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.7/90.7 kB 11.9 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 312.3/312.3 kB 33.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.3/51.3 kB 5.9 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.2/61.2 kB 7.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 kB 4.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.4/60.4 kB 6.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
!ls -l /content/Vim

total 44
drwxr-xr-x  2 root root  4096 Mar 30 12:50 assets
drwxr-xr-x  7 root root  4096 Mar 30 12:57 causal-conv1d
drwxr-xr-x 14 root root  4096 Mar 30 12:50 det
-rw-r--r--  1 root root 11348 Mar 30 12:50 LICENSE.txt
drwxr-xr-x 11 root root  4096 Mar 30 13:13 mamba-1p1p1
-rw-r--r--  1 root root  6284 Mar 30 12:50 README.md
drwxr-xr-x  7 root root  4096 Mar 30 12:50 seg
drwxr-xr-x  3 root root  4096 Mar 30 12:50 vim


In [ ]:
!ls -l /content/Vim/vim

total 132
-rw-r--r-- 1 root root  3703 Mar 30 12:50 augment.py
-rw-r--r-- 1 root root  4123 Mar 30 12:50 datasets.py
-rw-r--r-- 1 root root  4985 Mar 30 12:50 engine.py
-rw-r--r-- 1 root root   227 Mar 30 12:50 hubconf.py
-rw-r--r-- 1 root root 11354 Mar 30 12:50 LICENSE
-rw-r--r-- 1 root root  3388 Mar 30 12:50 losses.py
-rw-r--r-- 1 root root 25720 Mar 30 12:50 main.py
-rw-r--r-- 1 root root 26031 Mar 30 12:50 models_mamba.py
-rw-r--r-- 1 root root  5270 Mar 30 12:50 rope.py
-rw-r--r-- 1 root root  4075 Mar 30 12:50 run_with_submitit.py
-rw-r--r-- 1 root root  2584 Mar 30 12:50 samplers.py
drwxr-xr-x 2 root root  4096 Mar 30 12:50 scripts
-rw-r--r-- 1 root root  8445 Mar 30 12:50 utils.py
-rw-r--r-- 1 root root  2439 Mar 30 12:50 vim_requirements.txt


In [ ]:
# 🔍 DIAGNOSTIC: Check if mamba-ssm was actually built

print('='*60)
print('🔍 Diagnosing mamba-ssm installation')
print('='*60)

# 1. Check if package is registered with pip
print('\n[1/4] Checking pip packages...')
!pip list | grep -i mamba

# 2. Check if the build directory exists and has compiled files
print('\n[2/4] Checking for compiled CUDA extensions...')
!ls -la /content/Vim/mamba-1p1p1/build/lib*/mamba_ssm/ 2>/dev/null || echo "❌ No compiled files found"

# 3. Check if .so files (compiled libraries) exist
print('\n[3/4] Looking for .so files (compiled CUDA libraries)...')
!find /content/Vim/mamba-1p1p1 -name "*.so" -type f 2>/dev/null | head -5 || echo "❌ No .so files found"

# 4. Try to locate the mamba_ssm module
print('\n[4/4] Checking Python import paths...')
import sys
print(f'Python path includes:')
for p in sys.path[:5]:
    print(f'  - {p}')

print('\n' + '='*60)
print('📋 Diagnosis Summary:')
print('='*60)
print('If you see "No .so files found", the CUDA compilation failed.')
print('This means pip said "success" but the extensions never built.')
print('\n💡 Solution: Scroll up through the mamba-1p1p1 installation')
print('   output and look for lines containing "error:" or "failed"')

🔍 Diagnosing mamba-ssm installation

[1/4] Checking pip packages...
mamba_ssm                                1.1.1               /content/Vim/mamba-1p1p1

[2/4] Checking for compiled CUDA extensions...
❌ No compiled files found

[3/4] Looking for .so files (compiled CUDA libraries)...
/content/Vim/mamba-1p1p1/build/lib.linux-x86_64-cpython-312/selective_scan_cuda.cpython-312-x86_64-linux-gnu.so
/content/Vim/mamba-1p1p1/selective_scan_cuda.cpython-312-x86_64-linux-gnu.so

[4/4] Checking Python import paths...
Python path includes:
  - /content
  - /env/python
  - /usr/lib/python312.zip
  - /usr/lib/python3.12
  - /usr/lib/python3.12/lib-dynload

📋 Diagnosis Summary:
If you see "No .so files found", the CUDA compilation failed.
This means pip said "success" but the extensions never built.

💡 Solution: Scroll up through the mamba-1p1p1 installation
   output and look for lines containing "error:" or "failed"


In [ ]:
# Fix BOTH causal-conv1d AND mamba-ssm installations

%cd /content/Vim

print('🔧 Fixing causal-conv1d installation...')
!pip uninstall -y causal-conv1d
!pip install ./causal-conv1d -v
print('✅ causal-conv1d reinstalled\n')

print('🔧 Fixing mamba-ssm installation...')
!pip uninstall -y mamba-ssm
!pip install ./mamba-1p1p1 -v
print('✅ mamba-ssm reinstalled\n')

# Verify both imports work
print('='*60)
print('Verifying installations...')
print('='*60)

try:
    from causal_conv1d import causal_conv1d_fn
    print('✅ causal_conv1d imported successfully!')
except ImportError as e:
    print(f'❌ causal_conv1d failed: {e}')

try:
    from mamba_ssm import Mamba
    print('✅ mamba_ssm imported successfully!')
    print('\n🎉 ALL INSTALLATIONS COMPLETE!')
    print('▶️  Continue to next cells')
except ImportError as e:
    print(f'❌ mamba_ssm failed: {e}')

Streaming output truncated to the last 5000 lines.
  ptxas info    : Compiling entry function '_Z25selective_scan_bwd_kernelI32Selective_Scan_bwd_kernel_traitsILi32ELi4ELb0ELb1ELb0ELb1ELb1EN3c108BFloat16ENS1_7complexIfEEEEv12SSMParamsBwd' for 'sm_53'
  ptxas info    : Function properties for _Z25selective_scan_bwd_kernelI32Selective_Scan_bwd_kernel_traitsILi32ELi4ELb0ELb1ELb0ELb1ELb1EN3c108BFloat16ENS1_7complexIfEEEEv12SSMParamsBwd
      0 bytes stack frame, 0 bytes spill stores, 0 bytes spill loads
  ptxas info    : Used 161 registers, used 1 barriers, 664 bytes cmem[0], 48 bytes cmem[2]
  ptxas info    : Compile time = 385.030 ms
  ptxas info    : Compiling entry function '_Z25selective_scan_bwd_kernelI32Selective_Scan_bwd_kernel_traitsILi32ELi4ELb0ELb1ELb1ELb0ELb0EN3c108BFloat16ENS1_7complexIfEEEEv12SSMParamsBwd' for 'sm_53'
  ptxas info    : Function properties for _Z25selective_scan_bwd_kernelI32Selective_Scan_bwd_kernel_traitsILi32ELi4ELb0ELb1ELb1ELb0ELb0EN3c108BFloat16ENS1_7comp

/usr/local/lib/python3.12/dist-packages/mamba_ssm/ops/selective_scan_interface.py:163: FutureWarning: `torch.cuda.amp.custom_fwd(args...)` is deprecated. Please use `torch.amp.custom_fwd(args..., device_type='cuda')` instead.
  @custom_fwd
/usr/local/lib/python3.12/dist-packages/mamba_ssm/ops/selective_scan_interface.py:232: FutureWarning: `torch.cuda.amp.custom_bwd(args...)` is deprecated. Please use `torch.amp.custom_bwd(args..., device_type='cuda')` instead.
  @custom_bwd
/usr/local/lib/python3.12/dist-packages/mamba_ssm/ops/selective_scan_interface.py:300: FutureWarning: `torch.cuda.amp.custom_fwd(args...)` is deprecated. Please use `torch.amp.custom_fwd(args..., device_type='cuda')` instead.
  @custom_fwd
/usr/local/lib/python3.12/dist-packages/mamba_ssm/ops/selective_scan_interface.py:376: FutureWarning: `torch.cuda.amp.custom_bwd(args...)` is deprecated. Please use `torch.amp.custom_bwd(args..., device_type='cuda')` instead.
  @custom_bwd
/usr/local/lib/python3.12/dist-packages/

❌ mamba_ssm failed: cannot import name 'GreedySearchDecoderOnlyOutput' from 'transformers.generation' (/usr/local/lib/python3.12/dist-packages/transformers/generation/__init__.py)


In [ ]:
# Fix: Import only the core Mamba operations we need (avoid transformers dependency)

print('Testing minimal mamba_ssm import...')
try:
    # Import only the core Mamba block (no generation utilities)
    from mamba_ssm.modules.mamba_simple import Mamba
    print('✅ Mamba module imported successfully!')
    print('✅ Core functionality is working!')

    # Test it works
    import torch
    test_mamba = Mamba(d_model=128)
    print('✅ Mamba block instantiated successfully!')

    print('\n🎉 Installation complete and functional!')
    print('▶️  The transformers warning is NOT a problem for Vision Mamba detection')
    print('▶️  Continue to next cells')

except Exception as e:
    print(f'❌ Failed: {e}')
    print('\nTrying transformers fix...')
    # If still fails, downgrade transformers
    !pip install transformers==4.38.0 -q
    print('Retry import after transformers downgrade...')
    from mamba_ssm.modules.mamba_simple import Mamba
    print('✅ Success after transformers fix!')

Testing minimal mamba_ssm import...
✅ Mamba module imported successfully!
✅ Core functionality is working!
✅ Mamba block instantiated successfully!

🎉 Installation complete and functional!
▶️  The transformers warning is NOT a problem for Vision Mamba detection
▶️  Continue to next cells


In [ ]:
from models_mamba import VisionMamba

ModuleNotFoundError: No module named 'models_mamba'

## 3. Download Pretrained Vision Mamba Weights

**Model:** Vision Mamba-small (Vim-small)

- **Parameters:** 26M- **Source:** HuggingFace (hustvl/Vim-small-midclstok)

- **ImageNet Top-1 Accuracy:** 80.5%- **Depth:** 24 layers
- **Embedding Dimension:** 384

In [ ]:
# Download Vim-small pretrained weights from HuggingFace
print('📥 Downloading Vision Mamba-small pretrained weights...')
print('   Model: Vim-small (26M params, 80.5% ImageNet accuracy)')
print('   Source: HuggingFace Hub (hustvl/Vim-small-midclstok)\n')

# Install HuggingFace Hub
!pip install huggingface_hub -q

from huggingface_hub import hf_hub_download

# Download using HuggingFace Hub (supports caching and resume)
try:
    vim_ckpt = hf_hub_download(
        repo_id='hustvl/Vim-small-midclstok',
        filename='vim_s_midclstok_80p5acc.pth',
        cache_dir='/content/pretrained'
    )

    # Check file size
    file_size = os.path.getsize(vim_ckpt) / (1024**2)

    if file_size > 50:  # Should be ~100 MB
        print(f'✅ Pretrained weights downloaded successfully!')
        print(f'   Location: {vim_ckpt}')
        print(f'   Size: {file_size:.1f} MB')
        print(f'\n📊 These are the learned parameters from ImageNet-1K training')
        print(f'   They will be loaded into the VisionMamba architecture from Cell 8')
        print(f'\n▶️ Ready for backbone integration (next cell)')

        # Create symlink for easier access
        !ln -sf {vim_ckpt} /content/vim_small_pretrained.pth
        print(f'   Symlink created: /content/vim_small_pretrained.pth')
    else:
        print(f'⚠️  Downloaded file is too small ({file_size:.1f} MB)')
        print(f'   Expected: ~100 MB')
        raise ValueError('Download incomplete or corrupted')

except Exception as e:
    print(f'\n❌ HuggingFace download failed: {e}')
    print('\n📝 Manual download instructions:')
    print('   1. Visit: https://huggingface.co/hustvl/Vim-small-midclstok')
    print('   2. Download: vim_s_midclstok_80p5acc.pth')
    print('   3. Upload to Colab as: /content/vim_small_pretrained.pth')
    print('   4. Re-run this cell to verify')

📥 Downloading Vision Mamba-small pretrained weights...
   Model: Vim-small (26M params, 80.5% ImageNet accuracy)
   Source: HuggingFace Hub (hustvl/Vim-small-midclstok)



/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


vim_s_midclstok_80p5acc.pth:   0%|          | 0.00/413M [00:00<?, ?B/s]

✅ Pretrained weights downloaded successfully!
   Location: /content/pretrained/models--hustvl--Vim-small-midclstok/snapshots/babc4440f5fab6e08d97e371afa639c8cf98bf2c/vim_s_midclstok_80p5acc.pth
   Size: 394.2 MB

📊 These are the learned parameters from ImageNet-1K training
   They will be loaded into the VisionMamba architecture from Cell 8

▶️ Ready for backbone integration (next cell)
   Symlink created: /content/vim_small_pretrained.pth


## 4. Create Custom Vision Mamba Backbone for YOLO

**Multi-Scale Feature Extraction Strategy:**

Vision Mamba has 24 sequential blocks, all producing features of the same spatial resolution (H/16 × W/16) but with increasingly abstract representations. We extract features at three depths:

| Stage | Block | Spatial Size | Channels | YOLO Level | Receptive Field |
|-------|-------|--------------|----------|------------|-----------------|
| **Early** | 6 | H/16 × W/16 | 384 | P3 (stride 8) | Local details |
| **Middle** | 12 | H/16 × W/16 | 384 | P4 (stride 16) | Mid-level features |
| **Deep** | 18 | H/16 × W/16 | 384 | P5 (stride 32) | High-level semantics |

We then adapt these features to YOLO's expected channel dimensions [256, 512, 1024] using 1×1 convolutions.

In [ ]:
!pip install transformers==4.38.0 -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.1/131.1 kB 12.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.5/8.5 MB 131.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 54.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 107.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
sentence-transformers 5.3.0 requires transformers<6.0.0,>=4.41.0, but you have transformers 4.38.0 which is incompatible.


In [ ]:
# Quick verification after session restart
import sys
import os

sys.path.insert(0, '/content/Vim/vim')

# --- Start of Patch for models_mamba.py ---
# This patch comments out an unnecessary import from mamba_ssm.utils.generation
# which causes an ImportError due to a transformers version mismatch.
# GenerationMixin is for text generation and not required for the Vision Mamba backbone.
models_mamba_file_path = '/content/Vim/vim/models_mamba.py'

if os.path.exists(models_mamba_file_path):
    print(f"Attempting to patch '{models_mamba_file_path}' to remove GenerationMixin import...")
    with open(models_mamba_file_path, 'r') as f:
        lines = f.readlines()

    modified_lines = []
    patched = False
    for line in lines:
        if "from mamba_ssm.utils.generation import GenerationMixin" in line and not line.strip().startswith('#'):
            modified_lines.append(f"# {line.strip()}  # Patched by Colab to fix transformers ImportError\n")
            patched = True
        else:
            modified_lines.append(line)

    if patched:
        with open(models_mamba_file_path, 'w') as f:
            f.writelines(modified_lines)
        print(f"✅ Successfully patched '{models_mamba_file_path}'.")
    else:
        print(f"ℹ️ '{models_mamba_file_path}' already patched or original line not found.")
else:
    print(f"⚠️ Warning: '{models_mamba_file_path}' not found. Cannot patch.")
# --- End of Patch ---

# Test imports
from models_mamba import VisionMamba
from mamba_ssm.modules.mamba_simple import Mamba
print('✅ All imports successful!')
print('▶️ Ready to run VisionMambaBackbone cell')

Attempting to patch '/content/Vim/vim/models_mamba.py' to remove GenerationMixin import...
✅ Successfully patched '/content/Vim/vim/models_mamba.py'.
✅ All imports successful!
▶️ Ready to run VisionMambaBackbone cell


In [ ]:
import sys
import os
import torch
import torch.nn as nn
import math

# Add Vision Mamba to Python path
sys.path.insert(0, '/content/Vim/vim')

print('📦 Importing Vision Mamba modules...')
try:
    from models_mamba import VisionMamba
    print('✅ Vision Mamba modules imported successfully')
except ImportError as e:
    print(f'❌ Import failed: {e}')
    print('   Make sure Cell 9 (installation) completed successfully')
    raise

class VisionMambaBackbone(nn.Module):
    """
    Vision Mamba backbone adapted for YOLO object detection.

    Extracts multi-scale features from Vision Mamba-small at different depths
    and adapts them to YOLO's expected feature pyramid dimensions.

    Architecture:
        - Input: (B, 3, 640, 640) RGB image
        - Vision Mamba: 24 sequential blocks, bidirectional SSM
        - Feature extraction at blocks: 6, 12, 18
        - Output: [P3, P4, P5] features for YOLO FPN

    Args:
        pretrained_path (str): Path to ImageNet pretrained weights
        out_indices (tuple): Block indices to extract features from
        embed_dim (int): Vision Mamba embedding dimension (384 for Vim-small)
    """

    def __init__(self, pretrained_path=None, out_indices=(6, 12, 18), embed_dim=384):
        super().__init__()

        print('\n🏗️  Building Vision Mamba backbone...')

        # Create Vision Mamba-small model
        print('   Creating VisionMamba model (small variant)...')
        self.vim = VisionMamba(
            img_size=640,              # Input image size
            patch_size=16,             # Patch size (640/16 = 40x40 patches)
            depth=24,                  # 24 Mamba blocks
            embed_dim=384,             # Feature dimension (Vim-small)
            channels=3,                # RGB input
            num_classes=0,             # No classification head (backbone only)
            drop_rate=0.,              # No dropout
            drop_path_rate=0.1,        # Stochastic depth
            ssm_cfg=None,              # Default SSM config
            norm_epsilon=1e-5,         # Layer norm epsilon
            initializer_cfg=None,      # Default initialization
            fused_add_norm=True,       # Fused operations for speed
            rms_norm=True,             # RMS normalization
            residual_in_fp32=True,     # FP32 residuals for stability
            bimamba=True,              # ✅ BIDIRECTIONAL SCANNING (KEY FEATURE)
            pool_type='none'           # No pooling (keep spatial features)
        )

        # Load pretrained ImageNet weights
        if pretrained_path and os.path.exists(pretrained_path):
            print(f'   Loading pretrained weights from {pretrained_path}...')
            state_dict = torch.load(pretrained_path, map_location='cpu', weights_only=False)

            # Handle different checkpoint formats
            if 'model' in state_dict:
                state_dict = state_dict['model']
            elif 'state_dict' in state_dict:
                state_dict = state_dict['state_dict']

            # Remove classification head weights (we only need backbone)
            state_dict = {k: v for k, v in state_dict.items()
                         if not k.startswith('head') and not k.startswith('fc')}

            # Load weights
            # missing, unexpected = self.vim.load_state_dict(state_dict, strict=False)
            # Interpolate positional embeddings if image size differs
            if 'pos_embed' in state_dict:
                pos_embed_checkpoint = state_dict['pos_embed']
                embedding_size = pos_embed_checkpoint.shape[-1]
                num_patches = self.vim.patch_embed.num_patches  # For 640x640: 1600
                num_extra_tokens = self.vim.pos_embed.shape[-2] - num_patches

                # Original size from checkpoint (224x224 -> 14x14 = 196 patches)
                orig_size = int((pos_embed_checkpoint.shape[-2] - num_extra_tokens) ** 0.5)
                # New size (640x640 with patch_size 16 -> 40x40 = 1600 patches)
                new_size = int(num_patches ** 0.5)

                if orig_size != new_size:
                    print(f'   Position embedding: interpolating from {orig_size}x{orig_size} to {new_size}x{new_size}')
                    extra_tokens = pos_embed_checkpoint[:, :num_extra_tokens]
                    pos_tokens = pos_embed_checkpoint[:, num_extra_tokens:]
                    pos_tokens = pos_tokens.reshape(-1, orig_size, orig_size, embedding_size).permute(0, 3, 1, 2)
                    pos_tokens = torch.nn.functional.interpolate(
                        pos_tokens, size=(new_size, new_size), mode='bicubic', align_corners=False)
                    pos_tokens = pos_tokens.permute(0, 2, 3, 1).flatten(1, 2)
                    new_pos_embed = torch.cat((extra_tokens, pos_tokens), dim=1)
                    state_dict['pos_embed'] = new_pos_embed

            # Load weights
            missing, unexpected = self.vim.load_state_dict(state_dict, strict=False)

            print(f'   ✅ Loaded pretrained weights')
            if missing:
                # Filter out expected missing keys (head/fc layers)
                missing_filtered = [k for k in missing if not k.startswith('head') and not k.startswith('fc')]
                if missing_filtered:
                    print(f'   ⚠️  Missing keys (non-head): {len(missing_filtered)}')
            if unexpected:
                print(f'   ⚠️  Unexpected keys: {len(unexpected)}')
        else:
            print('   ⚠️  No pretrained weights loaded (training from scratch)')

        self.out_indices = out_indices  # Blocks to extract features from
        self.embed_dim = embed_dim      # 384 for Vim-small
        self.patch_size = 16            # Patch size

        print(f'\\n   📊 Configuration:')
        print(f'      Embedding dimension: {embed_dim}')
        print(f'      Depth (total blocks): 24')
        print(f'      Feature extraction at blocks: {out_indices}')
        print(f'      Bidirectional scanning: ✅ Enabled')

        # Feature dimension adapters (Vision Mamba → YOLO expected dims)
        # YOLO expects [256, 512, 1024] channels for P3, P4, P5 features
        print(f'\\n   🔧 Creating feature adapters...')
        self.feature_adapters = nn.ModuleList([
            nn.Conv2d(self.embed_dim, 256, 1),   # P3: 384 → 256 (stride 8)
            nn.Conv2d(self.embed_dim, 512, 1),   # P4: 384 → 512 (stride 16)
            nn.Conv2d(self.embed_dim, 1024, 1),  # P5: 384 → 1024 (stride 32)
        ])
        print(f'      P3 adapter: {embed_dim} → 256 channels')
        print(f'      P4 adapter: {embed_dim} → 512 channels')
        print(f'      P5 adapter: {embed_dim} → 1024 channels')

        # Register forward hooks to capture intermediate features
        self.features = {}
        self._register_hooks()

        print(f'\\n✅ Vision Mamba backbone created successfully!')

    def _register_hooks(self):
        """Register forward hooks to capture features at specified blocks"""
        def get_hook(name):
            def hook(module, input, output):
                self.features[name] = output
            return hook

        # Register hooks at specified layer indices
        for idx in self.out_indices:
            if hasattr(self.vim, 'layers') and idx < len(self.vim.layers):
                self.vim.layers[idx].register_forward_hook(get_hook(f'block_{idx}'))
            else:
                print(f'⚠️  Warning: Block {idx} not found in model')

    def forward(self, x):
        """
        Forward pass through Vision Mamba backbone.

        Args:
            x: (B, 3, H, W) input image tensor

        Returns:
            list of tensors: [P3, P4, P5] features for FPN
                P3: (B, 256, H/8, W/8)   - fine-grained features
                P4: (B, 512, H/16, W/16) - mid-level features
                P5: (B, 1024, H/32, W/32) - semantic features
        """
        B, C, H, W = x.shape

        # Clear previous features
        self.features = {}

        # Forward through Vision Mamba
        _ = self.vim(x)

        # Extract and process features at different depths
        feat_list = []
        for i, idx in enumerate(self.out_indices):
            if f'block_{idx}' in self.features:
                feat = self.features[f'block_{idx}']  # (B, N, C) where N = num_patches

                if isinstance(feat, tuple):
                  feat = feat[0]

                # Reshape from sequence to spatial: (B, N, C) → (B, C, H', W')
                # N = (H/patch_size) * (W/patch_size)
                H_feat = H // self.patch_size  # 640/16 = 40
                W_feat = W // self.patch_size  # 640/16 = 40

                # Remove extra tokens (class token, etc.) - keep only spatial tokens
                num_extra_tokens = feat.shape[1] - (H_feat * W_feat)
                if num_extra_tokens > 0:
                    feat = feat[:, num_extra_tokens:, :]  # Remove first N tokens

                # Transpose and reshape: (B, N, C) → (B, C, N) → (B, C, H', W')
                feat = feat.transpose(1, 2).reshape(B, self.embed_dim, H_feat, W_feat)

                # Adapt channel dimension to YOLO expected size
                feat = self.feature_adapters[i](feat)
                feat_list.append(feat)

        return feat_list  # [P3, P4, P5]

# Test the backbone
print('\\n' + '='*60)
print('🧪 Testing Vision Mamba backbone...')
print('='*60)

backbone = VisionMambaBackbone(
    pretrained_path='/content/vim_small_pretrained.pth',
    out_indices=(6, 12, 18),
    embed_dim=384
)
backbone.eval()

# Count parameters
total_params = sum(p.numel() for p in backbone.parameters())
trainable_params = sum(p.numel() for p in backbone.parameters() if p.requires_grad)

print(f'\\n📊 Model Statistics:')
print(f'   Total parameters: {total_params / 1e6:.2f}M')
print(f'   Trainable parameters: {trainable_params / 1e6:.2f}M')

# Test forward pass
print(f'\n🧪 Testing forward pass...')

# Move model and input to GPU (if available)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
backbone = backbone.to(device)
x_test = torch.randn(1, 3, 640, 640).to(device)

print(f'   Device: {device}')

with torch.no_grad():
    feats = backbone(x_test)

print(f'✅ Forward pass successful!')
print(f'\\n📦 Output features:')
for i, f in enumerate(feats):
    stride = 2 ** (i + 3)  # 8, 16, 32
    print(f'   P{i+3} (stride {stride:2d}): {f.shape} ({f.shape[1]} channels)')

print(f'\\n✅ Vision Mamba backbone ready for YOLO integration!')
print('▶️  Continue to next cell for training')

📦 Importing Vision Mamba modules...
✅ Vision Mamba modules imported successfully
\n============================================================
🧪 Testing Vision Mamba backbone...

🏗️  Building Vision Mamba backbone...
   Creating VisionMamba model (small variant)...
   Loading pretrained weights from /content/vim_small_pretrained.pth...
   Position embedding: interpolating from 14x14 to 40x40
   ✅ Loaded pretrained weights
\n   📊 Configuration:
      Embedding dimension: 384
      Depth (total blocks): 24
      Feature extraction at blocks: (6, 12, 18)
      Bidirectional scanning: ✅ Enabled
\n   🔧 Creating feature adapters...
      P3 adapter: 384 → 256 channels
      P4 adapter: 384 → 512 channels
      P5 adapter: 384 → 1024 channels
\n✅ Vision Mamba backbone created successfully!
\n📊 Model Statistics:
   Total parameters: 26.64M
   Trainable parameters: 26.64M

🧪 Testing forward pass...
   Device: cuda
✅ Forward pass successful!
\n📦 Output features:
   P3 (stride  8): torch.Size([1

## 5. YOLO Integration with Vision Mamba Backbone

**Integration Strategy:**

We'll build a complete detection model by combining:
1. **Vision Mamba backbone** (feature extraction) - already created ✓
2. **Feature Pyramid Network (FPN)** - multi-scale feature fusion
3. **Detection head** - bounding box + class prediction

**Two approaches available:**

### **Option A: Custom Detector (Full Control)**
Build the entire detector from scratch using our Vision Mamba backbone. Best for research and understanding the full pipeline.

### **Option B: Ultralytics Integration (Production Ready)**  
Integrate with Ultralytics YOLO framework. More complex but leverages optimized training pipeline.

**Recommendation:** Start with Option A for initial testing, then move to Option B for final training.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class SimpleDetectionHead(nn.Module):
    """
    Simplified YOLO-style detection head for Vision Mamba.

    Takes multi-scale features [P3, P4, P5] and produces:
    - Bounding boxes (4 values: x, y, w, h)
    - Objectness score (1 value)
    - Class probabilities (nc values)

    Total output per anchor: 4 + 1 + nc = 5 + nc
    """

    def __init__(self, nc=1, ch=(256, 512, 1024)):
        super().__init__()
        self.nc = nc  # Number of classes
        self.nl = len(ch)  # Number of detection layers
        self.no = nc + 5  # Number of outputs per anchor (box + obj + classes)

        # Detection heads for each scale
        self.m = nn.ModuleList([
            nn.Conv2d(ch[i], self.no, 1) for i in range(self.nl)
        ])

    def forward(self, x_list):
        """
        Args:
            x_list: [P3, P4, P5] features from backbone
                P3: (B, 256, H/8, W/8)
                P4: (B, 512, H/16, W/16)
                P5: (B, 1024, H/32, W/32)

        Returns:
            List of predictions at each scale
        """
        outputs = []
        for i in range(self.nl):
            # Apply detection head
            pred = self.m[i](x_list[i])  # (B, no, H, W)

            # Reshape for detection format
            B, _, H, W = pred.shape
            pred = pred.view(B, self.no, -1).transpose(1, 2).contiguous()

            outputs.append(pred)

        return outputs


class VisionMambaDetector(nn.Module):
    """
    Complete Vision Mamba detector combining:
    - Vision Mamba backbone (feature extraction)
    - Simple FPN (feature pyramid)
    - Detection head (bounding box prediction)
    """

    def __init__(self, nc=1, pretrained_path='/content/vim_small_pretrained.pth'):
        super().__init__()

        print('\n🏗️  Building Vision Mamba Detector...')

        # Backbone: Vision Mamba
        print('   [1/3] Creating Vision Mamba backbone...')
        self.backbone = VisionMambaBackbone(
            pretrained_path=pretrained_path,
            out_indices=(6, 12, 18),
            embed_dim=384
        )

        # Neck: Simple FPN (lateral connections + upsampling)
        print('   [2/3] Creating Feature Pyramid Network...')
        self.fpn_laterals = nn.ModuleList([
            nn.Conv2d(256, 256, 1),
            nn.Conv2d(512, 256, 1),
            nn.Conv2d(1024, 256, 1),
        ])

        self.fpn_outputs = nn.ModuleList([
            nn.Conv2d(256, 256, 3, padding=1),
            nn.Conv2d(256, 512, 3, padding=1),
            nn.Conv2d(256, 1024, 3, padding=1),
        ])

        # Head: Detection head
        print('   [3/3] Creating detection head...')
        self.head = SimpleDetectionHead(nc=nc, ch=(256, 512, 1024))

        print('\n✅ Vision Mamba Detector created successfully!')

        # Count parameters
        total_params = sum(p.numel() for p in self.parameters())
        print(f'   Total parameters: {total_params / 1e6:.2f}M')

    def forward(self, x):
        """
        Forward pass through entire detector.

        Args:
            x: (B, 3, H, W) input images

        Returns:
            List of predictions at each scale
        """
        # Extract features from backbone
        features = self.backbone(x)  # [P3, P4, P5]

        # Apply FPN (top-down pathway with lateral connections)
        fpn_features = []

        # Start from deepest layer
        prev_feat = None
        for i in range(len(features) - 1, -1, -1):
            # Lateral connection
            lateral = self.fpn_laterals[i](features[i])

            # Add upsampled previous layer
            if prev_feat is not None:
                if lateral.shape[-2:] != prev_feat.shape[-2:]:
                    prev_feat = F.interpolate(
                        prev_feat,
                        size=lateral.shape[-2:],
                        mode='nearest'
                    )
                lateral = lateral + prev_feat

            # Output convolution
            fpn_feat = self.fpn_outputs[i](lateral)
            fpn_features.insert(0, fpn_feat)

            prev_feat = lateral

        # Apply detection head
        predictions = self.head(fpn_features)

        return predictions


# Create the complete detector
print('\n' + '='*60)
print('🚀 Creating Complete Vision Mamba Detector')
print('='*60)

detector = VisionMambaDetector(
    nc=1,  # Number of classes (fire/smoke combined)
    pretrained_path='/content/vim_small_pretrained.pth'
)

# Test forward pass
print('\n🧪 Testing complete detector...')
device = 'cuda' if torch.cuda.is_available() else 'cpu'
detector = detector.to(device)
x_test = torch.randn(1, 3, 640, 640).to(device) # Moved x_test to the device

with torch.no_grad():
    preds = detector(x_test)

print('✅ Forward pass successful!')
print('\n📦 Detection outputs:')
for i, p in enumerate(preds):
    print(f'   Scale {i+1}: {p.shape} (num_predictions, 5+nc)')

print('\n✅ Vision Mamba Detector ready for training!')
print('▶️  Continue to next cell for training')



🚀 Creating Complete Vision Mamba Detector

🏗️  Building Vision Mamba Detector...
   [1/3] Creating Vision Mamba backbone...

🏗️  Building Vision Mamba backbone...
   Creating VisionMamba model (small variant)...
   Loading pretrained weights from /content/vim_small_pretrained.pth...
   Position embedding: interpolating from 14x14 to 40x40
   ✅ Loaded pretrained weights
\n   📊 Configuration:
      Embedding dimension: 384
      Depth (total blocks): 24
      Feature extraction at blocks: (6, 12, 18)
      Bidirectional scanning: ✅ Enabled
\n   🔧 Creating feature adapters...
      P3 adapter: 384 → 256 channels
      P4 adapter: 384 → 512 channels
      P5 adapter: 384 → 1024 channels
\n✅ Vision Mamba backbone created successfully!
   [2/3] Creating Feature Pyramid Network...
   [3/3] Creating detection head...

✅ Vision Mamba Detector created successfully!
   Total parameters: 31.24M

🧪 Testing complete detector...
✅ Forward pass successful!

📦 Detection outputs:
   Scale 1: torch.Size

## 6. Training with MMDetection (Recommended)

**Why MMDetection?**

MMDetection is the recommended approach for training Vision Mamba detector because:
- ✅ **Industry-standard framework** - Well-maintained by OpenMMLab
- ✅ **Easy custom backbone integration** - No source code modification needed
- ✅ **Complete training pipeline** - Handles data loading, loss, metrics automatically
- ✅ **Reproducible** - Configuration-based approach
- ✅ **Fair comparison** - Use same detection head for all backbones

**Complete Integration Guide:**

See the detailed MMDetection integration guide in:
📄 **`04_vision_mamba_mmdetection.md`**

**Quick Summary:**

1. Install MMDetection: `pip install mmdet`
2. Register Vision Mamba as custom backbone (copy VisionMambaBackbone class)
3. Create config file (`vim_yolo_boreal.py`)
4. Train with one command: `runner.train()`

**Expected Training Time:** 8-12 hours on A100 GPU

**Alternative**: Use custom training loop (requires ~300-400 lines of additional code)

**Output Files:**
- Checkpoint: `work_dirs/vim_yolo_boreal/best_coco/mAP_epoch_100.pth`
- Results CSV: Automatic mAP, precision, recall calculation
- Training logs: Complete logging and visualization

**Next Steps:**
1. Follow the MMDetection guide to set up training
2. Train the model (100 epochs)
3. Evaluate on test set
4. Compare results with YOLO and RT-DETR baselines

## 6.1 Install MMDectection




In [ ]:
# Install MMDetection framework
print('📦 Installing MMDetection and dependencies...')
print('   This will take 3-5 minutes\n')

# Fix for Python 3.12 compatibility: Install directly with pip instead of mim
# (mim has issues with pkgutil.ImpImporter in Python 3.12+)

# Step 1: Detect PyTorch and CUDA versions
import torch
torch_version = torch.__version__.split('+')[0]
cuda_version = torch.version.cuda

print(f'🔍 Detected environment:')
print(f'   PyTorch: {torch_version}')
print(f'   CUDA: {cuda_version}')

# Step 2: Install MMEngine
print(f'\n[1/3] Installing MMEngine...')
!pip install mmengine==0.10.3 -q
print('   ✓ MMEngine installed')

# Step 3: Install MMCV with smart version detection
print('\n[2/3] Installing MMCV (this takes 2-3 min)...')
print('   💡 Trying pre-built wheel first...')

# Map CUDA version to mmcv wheel URL
cuda_short = cuda_version.replace('.', '')[:4]  # e.g., "12.1" -> "121"
torch_short = torch_version.replace('.', '')[:3]  # e.g., "2.6.0" -> "260"

# Try multiple MMCV installation methods
mmcv_installed = False

# Method 1: Pre-built wheel for detected CUDA version
try:
    wheel_url = f'https://download.openmmlab.com/mmcv/dist/cu{cuda_short}/torch{torch_short}/index.html'
    print(f'   Trying: {wheel_url}')
    !pip install mmcv==2.1.0 -f {wheel_url} -q 2>&1 | grep -v "Requirement already satisfied" || true

    # Test import
    import mmcv
    mmcv_installed = True
    print('   ✓ MMCV installed (pre-built wheel)')
except:
    print('   ⚠️  Pre-built wheel not available')

# Method 2: Try older compatible version
if not mmcv_installed:
    print('   Trying MMCV 2.0.1...')
    try:
        !pip install mmcv==2.0.1 -q
        import mmcv
        mmcv_installed = True
        print('   ✓ MMCV 2.0.1 installed')
    except:
        pass

# Method 3: Install mmcv-lite (no CUDA ops, slower but works)
if not mmcv_installed:
    print('   Installing mmcv-lite (fallback, no CUDA acceleration)...')
    !pip install mmcv-lite==2.1.0 -q
    print('   ✓ mmcv-lite installed (training will be slower)')
    print('   ⚠️  Note: Using CPU version of MMCV ops')

# Step 4: Install MMDetection
print('\n[3/3] Installing MMDetection...')
!pip install mmdet==3.3.0 -q
print('   ✓ MMDetection installed')

# Step 5: Verify installation
print('\n✅ All packages installed!')
print('\n📦 Verifying installation...')

try:
    import mmdet
    import mmengine
    import mmcv

    print(f'   ✅ MMDetection: {mmdet.__version__}')
    print(f'   ✅ MMEngine: {mmengine.__version__}')
    print(f'   ✅ MMCV: {mmcv.__version__}')
    print('\n✅ All imports successful!')
    print('▶️ Ready for training setup!')
except ImportError as e:
    print(f'❌ Import failed: {e}')
    print('\n⚠️  Installation incomplete. Manual fix needed:')
    print('   1. Check Python version: python --version')
    print('   2. Try: pip install mmcv-lite')
    print('   3. Or use Kaggle Notebooks (better OpenMMLab support)')

📦 Installing MMDetection and dependencies...
   This will take 3-5 minutes

🔍 Detected environment:
   PyTorch: 2.10.0
   CUDA: 12.8

[1/3] Installing MMEngine...
   ✓ MMEngine installed

[2/3] Installing MMCV (this takes 2-3 min)...
   💡 Trying pre-built wheel first...
   Trying: https://download.openmmlab.com/mmcv/dist/cu128/torch210/index.html
  error: subprocess-exited-with-error
  
  × python setup.py egg_info did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
error: metadata-generation-failed

× Encountered error while generating package metadata.
╰─> See above for output.

note: This is an issue with the package mentioned above, not pip.
hint: See above for details.
   ⚠️  Pre-built wheel not available
   Trying MMCV 2.0.1...
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 469.5/469.5 kB 33.2 MB/s eta 0:00:00
  error: subprocess-exited-with-error
  
  × python setup.py egg_i

In [ ]:
import mmcv

## 6.2 Convert dataset to Coco format

In [39]:
import json
import yaml
from pathlib import Path
from PIL import Image
import sys

def yolo_to_coco(yolo_dir, split_name, class_names):
    """
    Convert YOLO format dataset to COCO format for MMDetection.

    Args:
        yolo_dir: Path to YOLO dataset directory
        split_name: 'train', 'val', or 'test'
        class_names: List of class names

    Returns:
        COCO format dictionary
    """
    images_dir = Path(yolo_dir) / 'images' / split_name
    labels_dir = Path(yolo_dir) / 'labels' / split_name

    # Check if directories exist
    if not images_dir.exists():
        print(f'   ⚠️  Images directory not found: {images_dir}')
        return None

    coco_data = {
        'images': [],
        'annotations': [],
        'categories': []
    }

    # Add categories
    for idx, name in enumerate(class_names):
        coco_data['categories'].append({
            'id': idx,
            'name': name,
            'supercategory': 'none'
        })

    # Get list of images
    image_files = sorted(images_dir.glob('*.jpg'))
    total_images = len(image_files)

    if total_images == 0:
        print(f'   ⚠️  No .jpg files found in {images_dir}')
        return None

    print(f'   Found {total_images} images')

    # Process images and annotations
    ann_id = 1
    for img_id, img_path in enumerate(image_files, 1):
        # Show progress every 100 images
        if ann_id <= 2000:
          print(img_id, img_path)
        if img_id % 100 == 0 or img_id == 1:
            print(f'   Processing {img_id}/{total_images}...', end='\r')
            sys.stdout.flush()

        try:
            # Load image to get dimensions
            img = Image.open(img_path)
            # print(f"First image size: {img.size}")
            # img.close()
            # print("First image loaded successfully.")
            width, height = img.size
            img.close()  # Close immediately to free memory

            # Add image info
            coco_data['images'].append({
                'id': img_id,
                'file_name': f'{split_name}/{img_path.name}',
                'width': width,
                'height': height
            })
            # print(f"length: {coco_data['images']}")

            # Load YOLO annotations
            label_path = labels_dir / f'{img_path.stem}.txt'
            if label_path.exists():
                with open(label_path) as f:
                    for line in f:
                        parts = line.strip().split()
                        if len(parts) >= 5:
                            class_id = int(parts[0])
                            cx, cy, w, h = map(float, parts[1:5])

                            # Convert YOLO (cx, cy, w, h - normalized) to COCO (x, y, w, h - absolute)
                            x = (cx - w / 2) * width
                            y = (cy - h / 2) * height
                            box_w = w * width
                            box_h = h * height

                            # Add annotation
                            coco_data['annotations'].append({
                                'id': ann_id,
                                'image_id': img_id,
                                'category_id': class_id,
                                'bbox': [x, y, box_w, box_h],
                                'area': box_w * box_h,
                                'iscrowd': 0
                            })
                            # print(f"ano - length: {coco_data['annotations']}")
                            ann_id += 1

        except Exception as e:
            print(f'\n   ⚠️  Error processing {img_path.name}: {e}')
            continue

    print()  # New line after progress
    return coco_data

# Load YAML to get class names
print('🔄 Converting YOLO dataset to COCO format...')
print('   (This shows progress every 100 images)\n')

with open(BOREAL_YAML) as f:
    yaml_data = yaml.safe_load(f)

class_names = yaml_data.get('names', ['smoke'])  # Default to 'fire' if not specified
data_root = Path(yaml_data['path'])

print(f'📂 Dataset root: {data_root}')
print(f'🏷️  Classes: {class_names}\n')

# Create COCO annotations directory
coco_dir = data_root / 'annotations'
coco_dir.mkdir(exist_ok=True)

# Convert train, val, test splits
import time
for split in ['train', 'val', 'test']:
    print(f'📝 Converting {split} split...')
    start_time = time.time()

    coco_data = yolo_to_coco(data_root, split, class_names)

    if coco_data is None:
        print(f'   ⚠️  Skipping {split} (no data found)\n')
        continue

    # Save COCO JSON
    output_path = coco_dir / f'{split}.json'
    with open(output_path, 'w') as f:
        json.dump(coco_data, f, indent=2)

    elapsed = time.time() - start_time
    print(f'   ✅ Saved to {output_path}')
    print(f'   Images: {len(coco_data["images"])}, Annotations: {len(coco_data["annotations"])}')
    print(f'   Time: {elapsed:.1f}s\n')

print(f'✅ Dataset conversion complete!')
print(f'📁 COCO annotations saved to: {coco_dir}')

🔄 Converting YOLO dataset to COCO format...
   (This shows progress every 100 images)

📂 Dataset root: /content/drive/MyDrive/thesis/data/boreal
🏷️  Classes: ['smoke']

📝 Converting train split...
   Found 2551 images
1 /content/drive/MyDrive/thesis/data/boreal/images/train/evoDJI_0001_frame1.jpg
2 /content/drive/MyDrive/thesis/data/boreal/images/train/evoDJI_0001_frame10.jpg
3 /content/drive/MyDrive/thesis/data/boreal/images/train/evoDJI_0001_frame100.jpg
4 /content/drive/MyDrive/thesis/data/boreal/images/train/evoDJI_0001_frame101.jpg
5 /content/drive/MyDrive/thesis/data/boreal/images/train/evoDJI_0001_frame102.jpg
6 /content/drive/MyDrive/thesis/data/boreal/images/train/evoDJI_0001_frame106.jpg
7 /content/drive/MyDrive/thesis/data/boreal/images/train/evoDJI_0001_frame107.jpg
8 /content/drive/MyDrive/thesis/data/boreal/images/train/evoDJI_0001_frame108.jpg
9 /content/drive/MyDrive/thesis/data/boreal/images/train/evoDJI_0001_frame109.jpg
10 /content/drive/MyDrive/thesis/data/boreal/im